<a href="https://colab.research.google.com/github/KoeJuntz/ResearchProject/blob/main/01_api_feasibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Kalshi API 10/16/2026

In [1]:
import requests
import pandas as pd

BASE = "https://external-api.kalshi.com/trade-api/v2"

def get_data(path, params=None):
    response = requests.get(
        BASE + path,
        params=params,
        timeout=30
    )
    response.raise_for_status()
    return response.json()

print(get_data("/historical/cutoff"))


{'market_positions_last_updated_ts': '2026-08-17T00:00:00Z', 'market_settled_ts': '2026-08-17T00:00:00Z', 'orders_updated_ts': '2026-10-02T00:00:00Z', 'trades_created_ts': '2026-08-17T00:00:00Z'}


In [2]:
markets = []
cursor = None

for page in range(20):
    params = {
        "series_ticker": "KXFEDDECISION",
        "limit": 100
    }

    if cursor:
        params["cursor"] = cursor

    result = get_data("/historical/markets", params)
    markets.extend(result["markets"])
    cursor = result.get("cursor")

    if not cursor:
        break

assert not cursor, "Inventory incomplete: pagination limit reached."

inventory = (
    pd.DataFrame(markets)
    .drop_duplicates("ticker")
)

print("Contracts:", len(inventory))
print("Event identifiers:", inventory["event_ticker"].nunique())

display(
    inventory.groupby("event_ticker")
    .agg(
        contracts=("ticker", "count"),
        close_time=("close_time", "min")
    )
    .sort_values("close_time")
)

Contracts: 140
Event identifiers: 28


,contracts,close_time
event_ticker,,
FEDDECISION-23MAY,6,2023-05-03T17:55:00Z
FEDDECISION-23JUN,5,2023-06-14T17:55:00Z
FEDDECISION-23JUL,5,2023-07-26T17:55:00Z
FEDDECISION-23SEP,5,2023-09-20T17:55:00Z
FEDDECISION-23NOV,5,2023-11-01T17:55:00Z
FEDDECISION-23DEC,5,2023-12-13T18:55:00Z
FEDDECISION-24JAN,5,2024-01-26T18:55:00Z
FEDDECISION-24JAN31,5,2024-01-31T18:55:00Z
FEDDECISION-24MAR20,5,2024-03-20T17:55:00Z


In [3]:
event_cutoffs = {
    "FEDDECISION-23MAY": "2023-05-02T16:00:00-04:00",
    "KXFEDDECISION-24DEC": "2024-12-17T16:00:00-05:00",
    "KXFEDDECISION-26JUL": "2026-07-28T16:00:00-04:00",
}

rows = []

for event, cutoff_text in event_cutoffs.items():
    cutoff = pd.Timestamp(cutoff_text).tz_convert("UTC")

    contracts = inventory.loc[
        inventory["event_ticker"].eq(event)
    ]

    assert not contracts.empty, f"Event missing: {event}"

    for _, contract in contracts.iterrows():
        ticker = contract["ticker"]

        result = get_data(
            f"/historical/markets/{ticker}/candlesticks",
            {
                "start_ts": int(
                    (cutoff - pd.Timedelta(days=7)).timestamp()
                ),
                "end_ts": int(cutoff.timestamp()),
                "period_interval": 60,
            }
        )

        candles = result.get("candlesticks", [])

        valid = [
            candle for candle in candles
            if candle["end_period_ts"] <= cutoff.timestamp()
            and candle.get("price", {}).get("close") is not None
        ]

        row = {
            "event": event,
            "contract": ticker,
            "outcome": contract.get("yes_sub_title"),
            "cutoff_utc": cutoff,
            "candles_retrieved": len(candles),
        }

        if valid:
            latest = max(valid, key=lambda c: c["end_period_ts"])

            timestamp = pd.to_datetime(
                latest["end_period_ts"], unit="s", utc=True
            )

            row.update({
                "candle_end_utc": timestamp,
                "yes_price_dollars": float(latest["price"]["close"]),
                "candle_volume": float(latest["volume"]),
                "candle_age_hours": (
                    cutoff - timestamp
                ).total_seconds() / 3600,
                "status": "retrieved",
            })
        else:
            row["status"] = "no priced candle in lookback"

        rows.append(row)

observations = pd.DataFrame(rows)

display(observations)

priced = observations.loc[
    observations["status"].eq("retrieved")
]

assert priced["yes_price_dollars"].between(0, 1).all()
assert priced["candle_age_hours"].ge(0).all()

print(
    "Events with retrieved observations:",
    priced["event"].nunique()
)

,event,contract,outcome,cutoff_utc,candles_retrieved,status,candle_end_utc,yes_price_dollars,candle_volume,candle_age_hours
0,FEDDECISION-23MAY,FEDDECISION-23MAY-TH50,Hike >50bps,2023-05-02 20:00:00+00:00,0,no priced candle in lookback,NaT,NaN,NaN,NaN
1,FEDDECISION-23MAY,FEDDECISION-23MAY-H50,Hike 50bps,2023-05-02 20:00:00+00:00,1,retrieved,2023-05-02 16:00:00+00:00,0.01,467.00,4.0
2,FEDDECISION-23MAY,FEDDECISION-23MAY-H25,Hike 25bps,2023-05-02 20:00:00+00:00,70,retrieved,2023-05-02 20:00:00+00:00,0.93,236.00,0.0
3,FEDDECISION-23MAY,FEDDECISION-23MAY-H0,No change,2023-05-02 20:00:00+00:00,59,retrieved,2023-05-02 20:00:00+00:00,0.06,207.00,0.0
4,FEDDECISION-23MAY,FEDDECISION-23MAY-TC25,Cut >25bps,2023-05-02 20:00:00+00:00,8,retrieved,2023-04-28 18:00:00+00:00,0.01,222.00,98.0
5,FEDDECISION-23MAY,FEDDECISION-23MAY-C25,Cut 25bps,2023-05-02 20:00:00+00:00,0,no priced candle in lookback,NaT,NaN,NaN,NaN
6,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H26,Hike >25bps,2024-12-17 21:00:00+00:00,2,retrieved,2024-12-12 07:00:00+00:00,0.01,4674.00,134.0
7,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H25,Hike 25bps,2024-12-17 21:00:00+00:00,9,retrieved,2024-12-17 08:00:00+00:00,0.01,100.00,13.0
8,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H0,No cut/hike,2024-12-17 21:00:00+00:00,104,retrieved,2024-12-17 21:00:00+00:00,0.03,7405.00,0.0
9,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-C25,Cut 25bps,2024-12-17 21:00:00+00:00,123,retrieved,2024-12-17 21:00:00+00:00,0.97,5464.00,0.0


Events with retrieved observations: 3


## Step 2 — Exact trade times and distribution quality
Run all cells from the top in this updated copy. The three new code cells below audit contract rules, retrieve trades, and summarize quality. They do not estimate dispersion or regressions.

A candle ending at the cutoff does not prove a trade occurred at that instant. We now retain actual trade timestamps. We do not replace missing prices with zero, normalize sums automatically, or assign point values to open-ended outcome buckets.

Official references: [historical trades](https://docs.kalshi.com/api-reference/historical/get-historical-trades), [historical market definitions](https://docs.kalshi.com/api-reference/historical/get-historical-market). Quote/candle/trade measurements must be identified separately.

In [4]:
# Cell 4: inspect the actual contract definitions already retrieved.
selected_markets = inventory.loc[
    inventory["event_ticker"].isin(event_cutoffs)
].copy()

selected_markets["open_ended_bucket"] = (
    selected_markets["yes_sub_title"].fillna("").str.contains(r">|<|at least|at most", case=False)
)
rule_columns = [
    "event_ticker", "ticker", "yes_sub_title",
    "open_ended_bucket", "rules_primary", "rules_secondary", "custom_strike"
]
display(selected_markets.reindex(columns=rule_columns))

print("Open-ended categories require further identification; no arbitrary bps values are assigned.")


,event_ticker,ticker,yes_sub_title,open_ended_bucket,rules_primary,rules_secondary,custom_strike
0,KXFEDDECISION-26JUL,KXFEDDECISION-26JUL-H26,Hike >25bps,True,If the Federal Reserve does a Hike of 25bps ...,"This market is mutually exclusive. Therefore, ...",{'Hike': '>25'}
1,KXFEDDECISION-26JUL,KXFEDDECISION-26JUL-H25,Hike 25bps,False,If the Federal Reserve does a Hike of 25bps on...,"This market is mutually exclusive. Therefore, ...",{'Hike': '25'}
2,KXFEDDECISION-26JUL,KXFEDDECISION-26JUL-H0,Fed maintains rate,False,If the Federal Reserve does a Hike of 0bps on ...,"This market is mutually exclusive. Therefore, ...",{'Hike': '0'}
3,KXFEDDECISION-26JUL,KXFEDDECISION-26JUL-C26,Cut >25bps,True,If the Federal Reserve does a Cut of 25bps o...,"This market is mutually exclusive. Therefore, ...",{'Cut': '>25'}
4,KXFEDDECISION-26JUL,KXFEDDECISION-26JUL-C25,Cut 25bps,False,If the Federal Reserve does a Cut of 25bps on ...,"This market is mutually exclusive. Therefore, ...",{'Cut': '25'}
65,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H26,Hike >25bps,True,If the Federal Reserve does a Hike of >25bps o...,"This market is mutually exclusive. Therefore, ...",{'Hike': '>25'}
66,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H25,Hike 25bps,False,If the Federal Reserve does a Hike of 25bps on...,"This market is mutually exclusive. Therefore, ...",{'Hike': '25'}
67,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H0,No cut/hike,False,If the Federal Reserve does a Hike of 0bps on ...,"This market is mutually exclusive. Therefore, ...",{'Hike': '0'}
68,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-C25,Cut 25bps,False,If the Federal Reserve does a Cut of 25bps on ...,"This market is mutually exclusive. Therefore, ...",{'Cut': '25'}
69,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-C26,Cut >25bps,True,If the Federal Reserve does a Cut of >25bps on...,"This market is mutually exclusive. Therefore, ...",{'Cut': '>25'}


Open-ended categories require further identification; no arbitrary bps values are assigned.


In [5]:
# Cell 5: exact historical trades, with bounded pagination and explicit failures.
import time

def checked_get(path, params=None):
    for attempt in range(3):
        r = requests.get(BASE + path, params=params, timeout=30)
        if r.status_code == 429 or r.status_code >= 500:
            if attempt < 2:
                try:
                    delay = float(r.headers.get("Retry-After", 2 ** (attempt + 1)))
                except ValueError:
                    delay = 2 ** (attempt + 1)
                time.sleep(min(max(delay, 1), 30))
                continue
        r.raise_for_status()
        return r.json()

archive = checked_get("/historical/cutoff")
trade_boundary = pd.Timestamp(archive["trades_created_ts"]).tz_convert("UTC")
request_log = []

def trade_window(ticker, start, end, page_cap=10):
    # Select tiers using the TRADE cutoff, not the market-settlement cutoff.
    paths = []
    if start < trade_boundary:
        paths.append("/historical/trades")
    if end >= trade_boundary:
        paths.append("/markets/trades")
    rows = []
    for path in paths:
        cursor = None
        seen = set()
        for page in range(page_cap):
            params = {
                "ticker": ticker,
                "min_ts": int(start.timestamp()),
                "max_ts": int(end.timestamp()),
                "limit": 1000
            }
            if cursor:
                params["cursor"] = cursor
            data = checked_get(path, params)
            request_log.append({
                "ticker": ticker, "path": path,
                "min_ts": params["min_ts"], "max_ts": params["max_ts"],
                "page": page + 1, "retrieved_utc": pd.Timestamp.now(tz="UTC").isoformat()
            })
            rows.extend(data["trades"])
            cursor = data.get("cursor")
            if not cursor:
                break
            if cursor in seen:
                raise RuntimeError("Repeated cursor: retrieval incomplete")
            seen.add(cursor)
        if cursor:
            raise RuntimeError(f"Page cap reached for {ticker}: retrieval incomplete")

    if not rows:
        return pd.DataFrame()
    trades = pd.DataFrame(rows).drop_duplicates("trade_id")
    assert trades["ticker"].eq(ticker).all(), "Unexpected ticker in response"
    trades["trade_time_utc"] = pd.to_datetime(trades["created_time"], utc=True)
    return trades.loc[
        trades["trade_time_utc"].between(start, end)
    ].sort_values(["trade_time_utc", "trade_id"])

exact_rows = []
for _, obs in observations.iterrows():
    ticker = obs["contract"]
    cutoff = obs["cutoff_utc"]
    lookback_start = cutoff - pd.Timedelta(days=7)
    # Start with the final priced candle's hour; extend through the cutoff.
    # If it yields no trades, search the full seven-day lookback.
    candle_end = obs.get("candle_end_utc")
    start = lookback_start
    if pd.notna(candle_end):
        start = max(lookback_start, candle_end - pd.Timedelta(hours=1))
    row = {
        "event": obs["event"], "contract": ticker, "outcome": obs["outcome"],
        "cutoff_utc": cutoff, "price_source": "last YES trade",
        "trade_time_utc": pd.NaT, "yes_price_dollars": float("nan"),
        "trade_size": float("nan"), "trade_age_hours": float("nan"),
        "carried_forward": None, "normalization": "none",
        "status": "pending"
    }
    try:
        trades = trade_window(ticker, start, cutoff)
        if trades.empty and start > lookback_start:
            start = lookback_start
            trades = trade_window(ticker, start, cutoff)
        row["query_start_utc"] = start
        if trades.empty:
            row["status"] = "no trade in seven-day lookback"
        else:
            last = trades.iloc[-1]
            price = float(last["yes_price_dollars"])
            assert 0 <= price <= 1, "Price outside dollar range"
            assert last["trade_time_utc"] <= cutoff, "Post-cutoff trade"
            latest = trades.loc[trades["trade_time_utc"].eq(last["trade_time_utc"])]
            row.update({
                "trade_time_utc": last["trade_time_utc"],
                "yes_price_dollars": price,
                "trade_size": float(last["count_fp"]),
                "trade_age_hours": (cutoff - last["trade_time_utc"]).total_seconds() / 3600,
                "carried_forward": last["trade_time_utc"] < cutoff,
                "is_block_trade": last.get("is_block_trade"),
                "latest_timestamp_price_tie": latest["yes_price_dollars"].nunique() > 1,
                "status": "retrieved"
            })
    except (requests.RequestException, ValueError, KeyError, RuntimeError, AssertionError) as exc:
        row["status"] = f"ERROR: {type(exc).__name__}: {exc}"
    exact_rows.append(row)
    print(f"{ticker}: {row['status']}", flush=True)

exact_observations = pd.DataFrame(exact_rows)
display(exact_observations)
print("Trade size is the selected transaction's size, NOT daily volume.")


FEDDECISION-23MAY-TH50: no trade in seven-day lookback
FEDDECISION-23MAY-H50: retrieved
FEDDECISION-23MAY-H25: retrieved
FEDDECISION-23MAY-H0: retrieved
FEDDECISION-23MAY-TC25: retrieved
FEDDECISION-23MAY-C25: no trade in seven-day lookback
KXFEDDECISION-24DEC-H26: retrieved
KXFEDDECISION-24DEC-H25: retrieved
KXFEDDECISION-24DEC-H0: retrieved
KXFEDDECISION-24DEC-C25: retrieved
KXFEDDECISION-24DEC-C26: retrieved
KXFEDDECISION-26JUL-H26: retrieved
KXFEDDECISION-26JUL-H25: retrieved
KXFEDDECISION-26JUL-H0: retrieved
KXFEDDECISION-26JUL-C26: retrieved
KXFEDDECISION-26JUL-C25: retrieved


,event,contract,outcome,cutoff_utc,price_source,trade_time_utc,yes_price_dollars,trade_size,trade_age_hours,carried_forward,normalization,status,query_start_utc,is_block_trade,latest_timestamp_price_tie
0,FEDDECISION-23MAY,FEDDECISION-23MAY-TH50,Hike >50bps,2023-05-02 20:00:00+00:00,last YES trade,NaT,NaN,NaN,NaN,None,none,no trade in seven-day lookback,2023-04-25 20:00:00+00:00,NaN,NaN
1,FEDDECISION-23MAY,FEDDECISION-23MAY-H50,Hike 50bps,2023-05-02 20:00:00+00:00,last YES trade,2023-05-02 15:16:55.456023+00:00,0.01,467.00,4.717929,True,none,retrieved,2023-05-02 15:00:00+00:00,False,False
2,FEDDECISION-23MAY,FEDDECISION-23MAY-H25,Hike 25bps,2023-05-02 20:00:00+00:00,last YES trade,2023-05-02 19:55:18.039996+00:00,0.93,8.00,0.078322,True,none,retrieved,2023-05-02 19:00:00+00:00,False,False
3,FEDDECISION-23MAY,FEDDECISION-23MAY-H0,No change,2023-05-02 20:00:00+00:00,last YES trade,2023-05-02 19:08:42.494902+00:00,0.06,80.00,0.854863,True,none,retrieved,2023-05-02 19:00:00+00:00,False,False
4,FEDDECISION-23MAY,FEDDECISION-23MAY-TC25,Cut >25bps,2023-05-02 20:00:00+00:00,last YES trade,2023-04-28 17:38:18.326526+00:00,0.01,5.00,98.361576,True,none,retrieved,2023-04-28 17:00:00+00:00,False,False
5,FEDDECISION-23MAY,FEDDECISION-23MAY-C25,Cut 25bps,2023-05-02 20:00:00+00:00,last YES trade,NaT,NaN,NaN,NaN,None,none,no trade in seven-day lookback,2023-04-25 20:00:00+00:00,NaN,NaN
6,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H26,Hike >25bps,2024-12-17 21:00:00+00:00,last YES trade,2024-12-12 06:12:58.542770+00:00,0.01,100.00,134.783738,True,none,retrieved,2024-12-12 06:00:00+00:00,False,False
7,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H25,Hike 25bps,2024-12-17 21:00:00+00:00,last YES trade,2024-12-17 07:31:27.417051+00:00,0.01,100.00,13.475717,True,none,retrieved,2024-12-17 07:00:00+00:00,False,False
8,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H0,No cut/hike,2024-12-17 21:00:00+00:00,last YES trade,2024-12-17 20:41:13.088628+00:00,0.03,1000.00,0.313031,True,none,retrieved,2024-12-17 20:00:00+00:00,False,False
9,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-C25,Cut 25bps,2024-12-17 21:00:00+00:00,last YES trade,2024-12-17 20:50:55.933233+00:00,0.97,108.00,0.151130,True,none,retrieved,2024-12-17 20:00:00+00:00,False,False


Trade size is the selected transaction's size, NOT daily volume.


In [6]:
# Cell 6: summarize coverage without disguising missing/stale observations.
quality_rows = []
for event, group in exact_observations.groupby("event", sort=False):
    expected = selected_markets.loc[selected_markets["event_ticker"].eq(event)]
    valid = group.loc[group["status"].eq("retrieved")]
    row = {
        "event": event,
        "expected_contracts": len(expected),
        "contracts_with_trades": len(valid),
        "missing_or_error": len(expected) - len(valid),
        "all_contracts_observed": len(valid) == len(expected),
        "sum_observed_prices": valid["yes_price_dollars"].sum(min_count=1),
        "oldest_trade_hours": valid["trade_age_hours"].max(),
        "open_ended_categories": int(expected["open_ended_bucket"].sum())
    }
    for hours in [6, 24, 48]:
        row[f"contracts_within_{hours}h"] = int(valid["trade_age_hours"].le(hours).sum())
    quality_rows.append(row)

quality_summary = pd.DataFrame(quality_rows)
display(quality_summary)

print("Price sums are diagnostics, not normalized probabilities.")
print("Incomplete events and open-ended categories are not ready for mean/dispersion calculations.")
print("Technical status remains PARTIAL PASS pending distributions and meeting-level coverage.")
print("Save this executed notebook back to the private GitHub repository.")


,event,expected_contracts,contracts_with_trades,missing_or_error,all_contracts_observed,sum_observed_prices,oldest_trade_hours,open_ended_categories,contracts_within_6h,contracts_within_24h,contracts_within_48h
0,FEDDECISION-23MAY,6,4,2,False,1.01,98.361576,2,3,3,3
1,KXFEDDECISION-24DEC,5,5,0,True,1.03,134.783738,2,3,4,4
2,KXFEDDECISION-26JUL,5,5,0,True,1.04,0.077076,2,5,5,5


Price sums are diagnostics, not normalized probabilities.
Incomplete events and open-ended categories are not ready for mean/dispersion calculations.
Technical status remains PARTIAL PASS pending distributions and meeting-level coverage.
Save this executed notebook back to the private GitHub repository.
